# Session 13 — Standard 20M TinyStories baseline

This is the first of the three assignment runs. It trains a decoder-only Transformer from random initialization on 50 million TinyStories tokens. The GPT-2 tokenizer is reused, but no pretrained model weights are loaded. The script prints the final validation loss, training throughput, peak CUDA memory, hardware, and exact run configuration.

## Run in Colab

1. Choose **Runtime → Change runtime type → GPU**. A T4 is a suitable starting point.
2. Run the cells in order. The first run downloads the public dataset and tokenizer and caches token IDs on Google Drive.
3. The training loop saves resumable checkpoints every 500 steps. If Colab disconnects, reconnect, rerun setup, and rerun the training cell; it resumes from the latest checkpoint.
4. Save the final printed results and `baseline_results.json` for the report. Training throughput excludes validation and checkpoint-save time; peak memory is CUDA allocated/reserved memory.

The baseline uses batch size 16, sequence length 256, and 19.86M model parameters. Change these only deliberately and record the actual values.

In [ ]:
%pip -q install "datasets>=3,<5" "transformers>=4.46,<5"

from pathlib import Path
import subprocess

repo = Path("/content/ERA_Portfolio")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/dhruv369/ERA_Portfolio.git", str(repo)],
        check=True,
    )
%cd /content/ERA_Portfolio/Assignments/S13

In [ ]:
import os
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Runtime → Change runtime type before training.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")

drive.mount("/content/drive")
os.environ["S13_RUN_DIR"] = "/content/drive/MyDrive/ERA_S13/baseline"

## First: validate the GPU and model

Run this short check first. It verifies the tokenizer download, exact parameter count, and model output shape without downloading TinyStories or starting training. If it passes, continue to the long baseline run.

In [ ]:
!python -u session13_baseline.py --smoke-only

## Then: train the baseline

This cell can run for several hours. Do not change the batch size after starting a run: the checkpoint records the configuration and will reject incompatible resumes. The script reports actual training tokens, validation loss, tokens/second, and CUDA peak memory.

In [ ]:
!python -u session13_baseline.py